# IFC nach RDF mit IFCtoLBD (LBD Level 1)

Dieses Notebook ist fuer die Ausfuehrung in **Renku** vorbereitet.

- IFC Quelle: `/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/Models/FoC_Preserving_IFC-Gantenbein.ifc`
- IFCtoLBD Fork: `/home/renku/work/IFCtoLBD`
- Ausgabeordner: `/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data`
- LBD Modell: **Level 1**

In [19]:
# Optional: nur falls in der Umgebung noch nicht vorhanden
!pip install --quiet rdflib


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from collections import deque
from datetime import datetime
from pathlib import Path
import shutil
import subprocess

# ===== Konfiguration (Renku Pfade) =====
ifc_input = Path("/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/Models/FoC_Preserving_IFC-Gantenbein.ifc")
ifctolbd_cli_jar = Path("/home/renku/work/dcaonnextcloud-500gb/ifc2lbd/IFCtoLBD_CLI.jar")
output_dir = Path("/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data")

# Optional: URI-Basis fuer erzeugte Ressourcen
base_uri = "http://dca.example.org/"

# Optional: Java Heap (bei grossen IFCs oft noetig)
java_heap = "8g"

# ===== Vorbedingungen mit klaren Fehlermeldungen =====
if shutil.which("java") is None:
    raise EnvironmentError(
        "Java wurde nicht gefunden. Bitte in Renku ein Java Runtime Environment (OpenJDK 17/21) bereitstellen."
    )

if not ifc_input.exists():
    raise FileNotFoundError(f"IFC-Datei nicht gefunden: {ifc_input}")

if not ifctolbd_cli_jar.exists():
    raise FileNotFoundError(f"IFCtoLBD_CLI.jar nicht gefunden: {ifctolbd_cli_jar}")

print(f"Verwende CLI-JAR: {ifctolbd_cli_jar}")

output_dir.mkdir(parents=True, exist_ok=True)

date_stamp = datetime.now().strftime("%Y%m%d")
output_file = output_dir / f"{ifc_input.stem}_{date_stamp}.ttl"

# ===== IFC -> RDF (LBD Level 1) via CLI =====
cmd = [
    "java",
    f"-Xmx{java_heap}",
    "-jar",
    str(ifctolbd_cli_jar),
    str(ifc_input),
    "--level",
    "1",
    "--target_file",
    str(output_file),
    "--url",
    base_uri,
]

print("Starte Konvertierung:")
print(" ".join(cmd))

# Live-Ausgabe waehrend der Laufzeit
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Wir behalten die letzten Zeilen fuer eine aussagekraeftige Fehlermeldung
log_tail = deque(maxlen=120)

for line in process.stdout:
    print(line, end="")
    log_tail.append(line.rstrip("\n"))

return_code = process.wait()

if return_code != 0:
    tail_text = "\n".join(log_tail) if log_tail else "(keine CLI-Ausgabe erfasst)"
    raise RuntimeError(
        "IFCtoLBD Konvertierung fehlgeschlagen.\n"
        f"Returncode: {return_code}\n"
        f"Kommando: {' '.join(cmd)}\n\n"
        "Letzte CLI-Ausgabe:\n"
        f"{tail_text}\n\n"
        "Hinweise:\n"
        "- Pruefe, ob das IFC intern defekte/inkonsistente Geometrien enthaelt.\n"
        "- Erhoehe java_heap (z. B. auf 12g oder 16g), falls Speicherproblem.\n"
        "- Teste denselben Befehl einmal direkt im Terminal fuer reproduzierbares Debugging."
    )

if not output_file.exists():
    raise FileNotFoundError(
        f"Konvertierung beendet, aber Ausgabedatei fehlt: {output_file}"
    )

print("\nKonvertierung erfolgreich.")
print(f"Ausgabe gespeichert: {output_file}")

Verwende CLI-JAR: /home/renku/work/dcaonnextcloud-500gb/ifc2lbd/IFCtoLBD_CLI.jar
Starte Konvertierung:
java -Xmx8g -jar /home/renku/work/dcaonnextcloud-500gb/ifc2lbd/IFCtoLBD_CLI.jar /home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/Models/FoC_Preserving_IFC-Gantenbein.ifc --level 1 --target_file /home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data/FoC_Preserving_IFC-Gantenbein_20260408.ttl --url http://dca.example.org/
Target is: /home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data/FoC_Preserving_IFC-Gantenbein_20260408.ttl
Using existing ifcOWL file
